# Lab 1 — Agentic RAG: Agente com 3 Ferramentas
## MBA em RAG & CAG Aplicados a Direito e Segurança Pública | Aula 10

**Objetivo:** Construir um agente RAG jurídico com 3 ferramentas distintas no LangGraph, simular uma pesquisa complexa com 4-6 tool calls e analisar o trace.

**Duração estimada:** 90 minutos

---

### Contexto

Um promotor de justiça precisa investigar um caso de lavagem de dinheiro com conexões internacionais. Ele precisa consultar:
1. Legislação aplicável (Lei 9.613/98, Lei 12.850/13)
2. Precedentes do STJ e STF sobre o tema
3. Ocorrências similares já registradas no banco de dados
4. Doutrina e análises recentes

Esse cenário requer **pelo menos 4 tool calls** para ser respondido completamente.

In [ ]:
# PASSO 1: Instalação e configuração
!pip install -q langgraph langchain langchain-openai langchain-community tavily-python

import os
import sqlite3
import time
import json
from typing import TypedDict, Annotated, Sequence
import operator

from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# Configurar API keys
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
except:
    pass

print("✅ Ambiente configurado")

In [ ]:
# PASSO 2: Preparar banco de dados jurídico
os.makedirs("../datasets", exist_ok=True)
DB_PATH = "../datasets/juridico_segpub.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS acordaos (
        id INTEGER PRIMARY KEY, numero TEXT, tribunal TEXT,
        data_julgamento TEXT, relator TEXT, tipo TEXT, ementa TEXT, resultado TEXT
    )
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS ocorrencias (
        id INTEGER PRIMARY KEY, numero_bo TEXT, data TEXT, tipo_crime TEXT,
        municipio TEXT, estado TEXT, status TEXT, descricao TEXT
    )
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS legislacao (
        id INTEGER PRIMARY KEY, numero TEXT, tipo TEXT,
        data_publicacao TEXT, ementa TEXT, artigos_principais TEXT
    )
""")

# Inserir dados de lavagem de dinheiro específicos para este lab
cursor.executemany("INSERT OR REPLACE INTO acordaos VALUES (?,?,?,?,?,?,?,?)", [
    (10, "RHC 150234", "STJ", "2023-08-22", "Min. Sebastião Reis Júnior",
     "Recurso em Habeas Corpus",
     "Lavagem de dinheiro. Operação Lava Jato. Prisão preventiva. Necessidade "
     "de fundamentação específica quanto à existência de perigo concreto.",
     "Provido"),
    (11, "AP 996", "STF", "2023-12-01", "Min. Edson Fachin",
     "Ação Penal",
     "Lavagem de capitais. Organização criminosa transnacional. Competência federal. "
     "Pena de 12 anos fixada considerando gravidade dos fatos e capacidade econômica.",
     "Procedente — Condenação 12 anos"),
    (12, "HC 567123", "STF", "2024-01-15", "Min. Alexandre de Moraes",
     "Habeas Corpus",
     "Lavagem de dinheiro. Bloqueio de ativos. Proporcionalidade. O bloqueio de "
     "ativos deve ser proporcional ao dano estimado ao erário.",
     "Ordem parcialmente concedida")
])

cursor.executemany("INSERT OR REPLACE INTO ocorrencias VALUES (?,?,?,?,?,?,?,?)", [
    (10, "BO-2024-010", "2024-01-15", "Lavagem de Dinheiro", "São Paulo", "SP",
     "Em investigação",
     "COAF identificou movimentações suspeitas de R$ 8,5M em 3 meses via contas laranja."),
    (11, "BO-2024-011", "2024-02-03", "Lavagem de Dinheiro", "Rio de Janeiro", "RJ",
     "Indiciado",
     "Suspeito utilizou criptomoedas para dissimular origem de R$ 2,3M provenientes "
     "de tráfico de drogas."),
    (12, "BO-2024-012", "2024-03-10", "Organização Criminosa", "Curitiba", "PR",
     "Preso em flagrante",
     "Grupo criminoso com 12 membros operava esquema de lavagem via imobiliárias.")
])

cursor.executemany("INSERT OR REPLACE INTO legislacao VALUES (?,?,?,?,?,?)", [
    (10, "Lei 9.613/1998", "Lei Federal", "1998-03-03",
     "Lei de Lavagem de Dinheiro. Dispõe sobre os crimes de lavagem ou ocultação de bens, "
     "direitos e valores; a prevenção da utilização do sistema financeiro para os ilícitos "
     "previstos nesta Lei; cria o COAF.",
     "Art. 1 - Crime de lavagem; Art. 9 - Obrigados a informar; Art. 14 - COAF"),
    (11, "Lei 12.850/2013", "Lei Federal", "2013-08-02",
     "Lei das Organizações Criminosas. Define organização criminosa e dispõe sobre a "
     "investigação criminal. Regulamenta colaboração premiada, infiltração de agentes.",
     "Art. 1 - Conceito ORCRIM; Art. 3 - Meios especiais; Art. 4 - Colaboração premiada")
])

conn.commit()
conn.close()
print(f"✅ Banco de dados preparado em {DB_PATH}")

In [ ]:
# PASSO 3: Definir as 3 ferramentas do agente

@tool
def search_docs(query: str) -> str:
    """Busca acórdãos, jurisprudência e doutrina jurídica.
    Use para: decisões judiciais, precedentes, institutos jurídicos.
    Args: query - termos de busca jurídica em português
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    palavras = query.lower().split()[:3]
    resultados = []
    for palavra in palavras:
        cursor.execute("""
            SELECT numero, tribunal, data_julgamento, ementa, resultado FROM acordaos
            WHERE LOWER(ementa) LIKE ?
        """, (f"%{palavra}%",))
        resultados.extend(cursor.fetchall())
    conn.close()
    vistos, unicos = set(), []
    for r in resultados:
        if r[0] not in vistos:
            vistos.add(r[0])
            unicos.append(r)
    if not unicos:
        return f"Sem resultados para: '{query}'"
    out = [f"Encontrados {len(unicos)} acórdãos:\n"]
    for num, trib, data, ementa, res in unicos[:3]:
        out.append(f"📋 {num} ({trib}, {data})\n   {ementa[:200]}\n   Resultado: {res}\n")
    return "\n".join(out)


@tool
def web_search(query: str) -> str:
    """Busca informações atuais na internet sobre direito e segurança pública.
    Use para: notícias recentes, artigos acadêmicos, dados atualizados.
    Args: query - termos de busca em português
    """
    tavily_key = os.getenv("TAVILY_API_KEY")
    if tavily_key:
        from tavily import TavilyClient
        client = TavilyClient(api_key=tavily_key)
        try:
            results = client.search(query, max_results=3)
            out = [f"Resultados web para '{query}':\n"]
            for r in results.get("results", []):
                out.append(f"🌐 {r['title']}\n   {r.get('content','')[:250]}\n")
            return "\n".join(out)
        except Exception as e:
            return f"Erro Tavily: {e}"
    return (
        f"[SIMULAÇÃO] Resultado para '{query}':\n"
        f"📰 'Lavagem de dinheiro via cripto cresce 40% em 2023' — Valor Econômico\n"
        f"   COAF reportou aumento significativo no uso de criptoativos para dissimular origem.\n"
        f"📰 'Senado aprova PL que amplia COAF' — Folha de SP\n"
        f"   Nova legislação aumenta obrigações de reporte para exchanges de cripto."
    )


@tool
def query_db(sql_query: str) -> str:
    """Executa consultas SQL em tabelas de acórdãos, ocorrências e legislação.
    Tabelas: acordaos, ocorrencias, legislacao
    Use para: dados estruturados, estatísticas, filtros específicos.
    Args: sql_query - query SELECT válida
    """
    if not sql_query.strip().upper().startswith("SELECT"):
        return "❌ Apenas SELECT permitido."
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        rows = conn.execute(sql_query).fetchall()
        conn.close()
        if not rows:
            return "Sem resultados."
        cols = rows[0].keys()
        out = [f"Colunas: {', '.join(cols)} | {len(rows)} registros\n"]
        for i, row in enumerate(rows[:5]):
            out.append(f"  [{i+1}] " + " | ".join(f"{c}: {str(row[c])[:80]}" for c in cols))
        return "\n".join(out)
    except Exception as e:
        return f"Erro SQL: {e}"


print("\n🔧 Ferramentas definidas:")
for t in [search_docs, web_search, query_db]:
    print(f"  ✓ {t.name}")

In [ ]:
# PASSO 4: Construir o agente com LangGraph

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    iteration_count: int

tools = [search_docs, web_search, query_db]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

MAX_ITER = 8  # Permitimos mais iterações para queries complexas

SYSTEM_PROMPT = """Você é um assistente jurídico especializado em direito penal e lavagem de dinheiro.
Para responder completamente, use as ferramentas na seguinte ordem lógica:
1. search_docs — busca precedentes e jurisprudência relacionados
2. query_db — busca ocorrências e dados estruturados no banco
3. query_db — busca legislação aplicável
4. web_search — se precisar de informações mais recentes
Sempre justifique cada tool call e cite as fontes na resposta final."""

def agent_node(state: AgentState) -> dict:
    iteration = state.get("iteration_count", 0)
    if iteration >= MAX_ITER:
        return {
            "messages": [AIMessage(content=f"[GUARD] Máximo de {MAX_ITER} iterações atingido.")],
            "iteration_count": iteration
        }
    all_msgs = [SystemMessage(content=SYSTEM_PROMPT)] + list(state["messages"])
    response = llm_with_tools.invoke(all_msgs)
    return {"messages": [response], "iteration_count": iteration + 1}

def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"

tool_node = ToolNode(tools)
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
workflow.add_edge("tools", "agent")
app = workflow.compile()

print("✅ Agente compilado com sucesso!")
print(f"   MAX_ITER = {MAX_ITER}")
print(f"   Ferramentas: {[t.name for t in tools]}")

In [ ]:
# PASSO 5: Executar pesquisa complexa (4-6 tool calls)

QUERY_COMPLEXA = """
Faça uma pesquisa completa sobre lavagem de dinheiro via criptomoedas no Brasil:
1. Quais precedentes do STF e STJ existem sobre o tema?
2. Há ocorrências registradas no banco de dados envolvendo cripto?
3. Qual a legislação aplicável (Lei 9.613 e Lei 12.850)?
4. Há desenvolvimentos recentes na legislação ou jurisprudência?
Consolide tudo em um parecer jurídico estruturado.
"""

def executar_pesquisa_complexa(query: str) -> dict:
    """Executa o agente e coleta métricas detalhadas."""
    inicio = time.time()
    inputs = {"messages": [HumanMessage(content=query)], "iteration_count": 0}
    
    trace = []  # Registra cada passo
    resposta_final = ""
    tool_calls_total = 0
    
    print("🚀 INICIANDO PESQUISA COMPLEXA")
    print("=" * 70)
    print(f"Query: {query[:200]}...")
    print("=" * 70)
    
    for step_num, event in enumerate(app.stream(inputs, {"recursion_limit": 20})):
        for node_name, node_output in event.items():
            msgs = node_output.get("messages", [])
            for msg in msgs:
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        tool_calls_total += 1
                        trace.append({
                            "step": step_num,
                            "tipo": "tool_call",
                            "ferramenta": tc["name"],
                            "args": str(tc.get("args", {}))[:100]
                        })
                        print(f"\n  ⚡ Tool call #{tool_calls_total}: {tc['name']}")
                        print(f"     Args: {str(tc.get('args',''))[:80]}")
                
                elif isinstance(msg, ToolMessage):
                    trace.append({
                        "step": step_num,
                        "tipo": "observation",
                        "conteudo": str(msg.content)[:150]
                    })
                    print(f"     → Obs: {str(msg.content)[:120]}")
                
                elif hasattr(msg, "content") and isinstance(msg.content, str):
                    if len(msg.content) > 100:
                        resposta_final = msg.content
    
    duracao = time.time() - inicio
    
    print(f"\n{'='*70}")
    print(f"✅ PESQUISA CONCLUÍDA")
    print(f"{'='*70}")
    print(f"⏱️  Duração total: {duracao:.1f}s")
    print(f"🔧 Total de tool calls: {tool_calls_total}")
    print(f"📊 Ferramentas usadas: {list(set(t['ferramenta'] for t in trace if t['tipo']=='tool_call'))}")
    
    print(f"\n{'='*70}")
    print("📄 RESPOSTA FINAL:")
    print(f"{'='*70}")
    print(resposta_final[:2000] if resposta_final else "[Resposta não capturada]")
    
    return {
        "trace": trace,
        "resposta": resposta_final,
        "duracao": duracao,
        "tool_calls": tool_calls_total
    }


if os.getenv("OPENAI_API_KEY"):
    resultado = executar_pesquisa_complexa(QUERY_COMPLEXA)
else:
    print("ℹ️  Configure OPENAI_API_KEY para executar o Lab 1 completo.")
    print("   O código está pronto — adicione sua chave e re-execute.")

In [ ]:
# PASSO 6: Análise do trace e identificação de padrões

def analisar_trace(trace: list) -> None:
    """Analisa o trace do agente buscando padrões e ineficiências."""
    tool_calls = [t for t in trace if t["tipo"] == "tool_call"]
    
    print("\n📊 ANÁLISE DO TRACE DO AGENTE")
    print("=" * 50)
    
    # Distribuição de ferramentas
    from collections import Counter
    dist = Counter(t["ferramenta"] for t in tool_calls)
    print("\n📈 Distribuição de tool calls:")
    for ferramenta, count in dist.most_common():
        barra = "█" * count
        print(f"  {ferramenta:<25} {barra} ({count}x)")
    
    # Detecta redundâncias
    ferramentas_lista = [t["ferramenta"] for t in tool_calls]
    repetidas = [f for f, c in dist.items() if c > 1]
    if repetidas:
        print(f"\n⚠️  Ferramentas chamadas múltiplas vezes: {repetidas}")
        print("   → Revisar prompt do sistema para reduzir redundância")
    else:
        print("\n✅ Sem redundâncias detectadas — trajetória eficiente")
    
    # Sequência de chamadas
    print(f"\n📋 Sequência de execução:")
    for i, tc in enumerate(tool_calls, 1):
        print(f"  {i}. {tc['ferramenta']} → {tc['args'][:60]}")


# Análise com trace simulado se não tiver executado o agente
trace_simulado = [
    {"step": 1, "tipo": "tool_call", "ferramenta": "search_docs", "args": "lavagem dinheiro STF STJ"},
    {"step": 2, "tipo": "observation", "conteudo": "3 acórdãos encontrados"},
    {"step": 3, "tipo": "tool_call", "ferramenta": "query_db", "args": "SELECT * FROM ocorrencias WHERE tipo_crime LIKE '%lavagem%'"},
    {"step": 4, "tipo": "observation", "conteudo": "2 registros encontrados"},
    {"step": 5, "tipo": "tool_call", "ferramenta": "query_db", "args": "SELECT * FROM legislacao WHERE LOWER(ementa) LIKE '%lavagem%'"},
    {"step": 6, "tipo": "observation", "conteudo": "Lei 9.613/1998 e Lei 12.850/2013 encontradas"},
    {"step": 7, "tipo": "tool_call", "ferramenta": "web_search", "args": "lavagem dinheiro criptomoedas legislação 2024"},
    {"step": 8, "tipo": "observation", "conteudo": "Artigos recentes sobre PL de regulação de criptos"}
]

if 'resultado' in dir() and resultado.get('trace'):
    analisar_trace(resultado['trace'])
else:
    print("📊 Análise com trace simulado (execute o agente para análise real):")
    analisar_trace(trace_simulado)

---
## ✏️ Exercício

**Tarefa:** Modifique o `SYSTEM_PROMPT` do agente para:
1. Reduzir o número de tool calls redundantes
2. Melhorar a ordem lógica das buscas
3. Garantir que a resposta final cite pelo menos 3 fontes

Execute novamente e compare os traces antes e depois da modificação.

**Critério de sucesso:** Resposta completa com ≤ 5 tool calls.